# Silver Layer — CRM: Customer Info

Transforms the bronze `workspace.bronze.cust_info` table into `workspace.silver.customer_info` with meaningful, readable column names.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (e.g., `cst_id` → `customer_id`, `cst_firstname` → `first_name`) |
| 2 | **Trim Whitespace** | Trimmed leading/trailing spaces from `first_name` and `last_name` |
| 3 | **Full Name Column** | Created new `full_name` column by concatenating trimmed `first_name` + space + trimmed `last_name` |
| 4 | **Marital Status Mapping** | Mapped codes to full words: `M` → `Married`, `S` → `Single` |
| 5 | **Gender Mapping** | Mapped codes to full words: `M` → `Male`, `F` → `Female` |

---

## Column Mapping (Bronze → Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `cst_id` | `customer_id` | Renamed |
| `cst_key` | `customer_key` | Renamed |
| `cst_firstname` | `first_name` | Renamed + Trimmed |
| `cst_lastname` | `last_name` | Renamed + Trimmed |
| — | `full_name` | New column: `TRIM(first_name) + ' ' + TRIM(last_name)` |
| `cst_marital_status` | `marital_status` | Renamed + Mapped (`M`→`Married`, `S`→`Single`) |
| `cst_gndr` | `gender` | Renamed + Mapped (`M`→`Male`, `F`→`Female`) |
| `cst_create_date` | `create_date` | Renamed |

---

## Implementation

All transformations are implemented in **Spark SQL**:
- `TRIM()` — remove leading/trailing whitespace
- `CONCAT()` — build the `full_name` column
- `CASE WHEN` — conditional mapping for marital_status and gender
- `CREATE OR REPLACE TABLE ... AS SELECT` — write to silver layer

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.customer_info AS
SELECT
    cst_id                              AS customer_id,
    cst_key                             AS customer_key,
    TRIM(cst_firstname)                 AS first_name,
    TRIM(cst_lastname)                  AS last_name,
    CONCAT(TRIM(cst_firstname), ' ', TRIM(cst_lastname)) AS full_name,
    CASE
        WHEN cst_marital_status = 'M' THEN 'Married'
        WHEN cst_marital_status = 'S' THEN 'Single'
        ELSE cst_marital_status
    END                                 AS marital_status,
    CASE
        WHEN cst_gndr = 'M' THEN 'Male'
        WHEN cst_gndr = 'F' THEN 'Female'
        ELSE cst_gndr
    END                                 AS gender,
    cst_create_date                     AS create_date
FROM workspace.bronze.cust_info

In [0]:
%sql
SELECT * FROM workspace.silver.customer_info LIMIT 10